# Ejemplo Básico del Cálculo de la Matriz de Admitancias

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Número de nodos
n=5

# Crear un dataframe para los datos de las ramas
data = {
    "Bus_from": [2, 2, 4, 1, 3],
    "Bus_to": [4, 5, 5, 5, 4],
    "R_pu": [0.0090, 0.0045, 0.00225, 0.00150, 0.00075],
    "X_pu": [0.100, 0.05, 0.025, 0.02, 0.01],
    "G_pu": [0, 0, 0, 0, 0],
    "B_pu": [1.72, 0.88, 0.44, 0, 0],
    "Tap": [None, None, None, None, None]
}


ramas = pd.DataFrame(data)
print(ramas)


   Bus_from  Bus_to     R_pu   X_pu  G_pu  B_pu   Tap
0         2       4  0.00900  0.100     0  1.72  None
1         2       5  0.00450  0.050     0  0.88  None
2         4       5  0.00225  0.025     0  0.44  None
3         1       5  0.00150  0.020     0  0.00  None
4         3       4  0.00075  0.010     0  0.00  None


In [ ]:
# Método básico para construir Ybus

def build_ybus(datos, nbuses):
    Ybus = np.zeros((nbuses, nbuses), dtype=complex)

    for i in range(len(datos)):
        k = int(datos[i,0])-1  # Nodo "from" convertido a índice 0-based
        m = int(datos[i,1])-1  # Nodo "to" convertido a índice 0-based

        R = datos[i,2]  # extraer la resistencia en serie
        X = datos[i,3]  # extraer la reactancia en serie
        G = datos[i,4]  # extraer la conductancia
        B = datos[i,5]  # extraer la susceptancia

        y_serie = 1/complex(R, X)  # Admitancia en serie
        y_shunt = complex(G,B)/2    # Susceptancia shunt por línea

        # Elementos de la diagonal
        Ybus[k,k]+=y_serie + y_shunt
        Ybus[m,m]+=y_serie + y_shunt

        # Elementos fuera de diagonal
        Ybus[k,m]-=y_serie
        Ybus[m,k]-=y_serie

    return Ybus

# Aplicación del método
Ybus=build_ybus(ramas.values, n)

# Configurar opciones de impresión para mejor visualización de matrices
np.set_printoptions(precision=4, suppress=True, linewidth=100)
print(Ybus)

[[ 3.729  -49.7203j  0.      +0.j      0.      +0.j      0.      +0.j     -3.729  +49.7203j]
 [ 0.      +0.j      2.6783 -28.459j   0.      +0.j     -0.8928  +9.9197j -1.7855 +19.8393j]
 [ 0.      +0.j      0.      +0.j      7.458  -99.4406j -7.458  +99.4406j  0.      +0.j    ]
 [ 0.      +0.j     -0.8928  +9.9197j -7.458  +99.4406j 11.9219-147.9589j -3.5711 +39.6786j]
 [-3.729  +49.7203j -1.7855 +19.8393j  0.      +0.j     -3.5711 +39.6786j  9.0856-108.5782j]]


#Cálculo de Zbus

In [ ]:
Zbus = np.linalg.inv(Ybus)
print(Zbus)

[[ 0.0026-0.2974j -0.0006-0.3355j  0.0001-0.3281j  0.0001-0.3281j  0.0011-0.3174j]
 [-0.0006-0.3355j  0.0011-0.3171j -0.0009-0.3392j -0.0009-0.3392j -0.0006-0.3355j]
 [ 0.0001-0.3281j -0.0009-0.3392j  0.0018-0.3072j  0.0011-0.3172j  0.0001-0.3281j]
 [ 0.0001-0.3281j -0.0009-0.3392j  0.0011-0.3172j  0.0011-0.3172j  0.0001-0.3281j]
 [ 0.0011-0.3174j -0.0006-0.3355j  0.0001-0.3281j  0.0001-0.3281j  0.0011-0.3174j]]


In [ ]:
# Método para reducción de Kron

def kron_reduction(Ybus, keep_nodes):
    """
    Aplica reducción de Kron a la matriz Ybus.

    Parámetros:
        Ybus : np.ndarray
            Matriz Ybus (n x n) compleja.
        keep_nodes : lista[int]
            Índices (0-based) de los nodos que se conservarán (frontera).

    Retorna:
        Yred : np.ndarray
            Matriz Ybus reducida a los nodos en keep_nodes.
    """
    n = Ybus.shape[0]
    all_nodes = np.arange(n)
    eliminate_nodes = np.setdiff1d(all_nodes, keep_nodes)

    # Partición de la matriz
    Yaa = Ybus[np.ix_(keep_nodes, keep_nodes)]
    Yab = Ybus[np.ix_(keep_nodes, eliminate_nodes)]
    Yba = Ybus[np.ix_(eliminate_nodes, keep_nodes)]
    Ybb = Ybus[np.ix_(eliminate_nodes, eliminate_nodes)]

    # Reducción de Kron: Yred = Yaa - Yab * inv(Ybb) * Yba
    Yred = Yaa - Yab @ np.linalg.inv(Ybb) @ Yba

    return Yred

# Aplicación del método
keep_nodes = [0, 1, 2, 3]
Yred = kron_reduction(Ybus, keep_nodes)

print(np.round(Yred, 4))

[[ 2.2189 -26.954j  -0.7388  +9.0853j  0.      +0.j     -1.4775 +18.1707j]
 [-0.7388  +9.0853j  2.3291 -24.8341j  0.      +0.j     -1.5911 +17.1694j]
 [ 0.      +0.j      0.      +0.j      7.458  -99.4406j -7.458  +99.4406j]
 [-1.4775 +18.1707j -1.5911 +17.1694j -7.458  +99.4406j 10.5252-133.4594j]]
